In [2]:
from fastapi import FastAPI
from pydantic import BaseModel, Field
from sqlalchemy import URL, create_engine, Column, Integer, String
from sqlalchemy.orm import DeclarativeBase, sessionmaker

from dotenv import load_dotenv
from typing import Literal
import os

In [3]:
app = FastAPI()
load_dotenv()

True

In [4]:
DATABASE_URL = URL.create(
    drivername="postgresql+psycopg2",
    username=os.environ["POSTGRES_USER"],
    password=os.environ["POSTGRES_PASSWORD"],
    host=os.environ["POSTGRES_HOST"],
    port=int(os.environ["POSTGRES_PORT"]),
    database=os.environ["POSTGRES_DB"],
)
engine = create_engine(DATABASE_URL)
Session = sessionmaker(bind=engine)

class Base(DeclarativeBase):
    pass

In [5]:
class add_row(BaseModel):
    id: int = Field(..., description="The ID of the data")
    gender: Literal["Men", "Women"] = Field(..., description="The gender of the clothing item")
    master_category: str = Field(..., description="The master category of the clothing item")
    article_type: str = Field(..., description="The article type of the clothing item")
    base_color: str = Field(..., description="The base color of the clothing item")
    season: Literal["Summer", "Winter", "Fall", "Spring"] = Field(..., description="The season for which the clothing item is suitable")
    year: int = Field(..., description="The year of the clothing item")
    usage: Literal["Casual", "Formal", "Sports", "Party", "Ethnic"] = Field(..., description="The usage of the clothing item")
    product_display_name: str = Field(..., description="The display name of the clothing item")


class update_data(BaseModel):
    id: int = Field(..., description="The ID of the data")
    gender: Literal["Men", "Women"] = Field(..., description="The gender of the clothing item")
    master_category: str = Field(..., description="The master category of the clothing item")
    article_type: str = Field(..., description="The article type of the clothing item")
    base_color: str = Field(..., description="The base color of the clothing item")
    season: Literal["Summer", "Winter", "Fall", "Spring"] = Field(..., description="The season for which the clothing item is suitable")
    year: int = Field(..., description="The year of the clothing item")
    usage: Literal["Casual", "Formal", "Sports", "Party", "Ethnic"] = Field(..., description="The usage of the clothing item")
    product_display_name: str = Field(..., description="The display name of the clothing item")


class delete_data(BaseModel):
    id: int = Field(..., description="The ID of the data to be deleted")


class GetDataRequest(BaseModel):
    id: int = Field(..., description="The ID of the data to be retrieved")

In [6]:
class db_table(Base):
    __tablename__ = "DATA_TABLE"
    __table_args__ = {"schema": "public"}

    id = Column(Integer, primary_key=True, index=True)
    gender = Column(String, nullable=False)
    master_category = Column("masterCategory", String, nullable=False)
    sub_category = Column("subCategory", String, nullable=True)
    article_type = Column("articleType", String, nullable=False)
    base_color = Column("baseColour", String, nullable=False)
    season = Column(String, nullable=False)
    year = Column(Integer, nullable=False)
    usage = Column(String, nullable=False)
    product_display_name = Column("productDisplayName", String, nullable=False)

In [7]:
@app.get("/get_data")
def get_data():
    with Session() as session:
        result = session.query(db_table).all()
        if result:
            return [{
                "id": result.id,
                "gender": result.gender,
                "base_color": result.base_color,
                "product_display_name": result.product_display_name
            } for result in result]
        
        return {"error": "Data not found"}


@app.get("/")
def home():
    return {"message": "Welcome to the FastAPI!"}


@app.get("/get_data/{id}")
def get_data_by_id(id:int):
    with Session() as session:
        result = session.query(db_table).filter(db_table.id == id).first()
        if result:
            return {
                "id": result.id,
                "gender": result.gender,
                "base_color": result.base_color,
                "product_display_name": result.product_display_name
            }
        return {"error": "Data not found"}


@app.post("/add_data")
def add_data(data: add_row):
    with Session() as session:
        new_row = db_table(
            id=data.id,
            gender=data.gender,
            master_category=data.master_category,
            article_type=data.article_type,
            base_color=data.base_color,
            season=data.season,
            year=data.year,
            usage=data.usage,
            product_display_name=data.product_display_name
        )
        session.add(new_row)
        session.commit()
        return {"message": "Data added successfully"}

In [8]:
import threading
import time
import uvicorn

server = uvicorn.Server(
    uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info")
)
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()

for _ in range(50):
    if server.started:
        break
    time.sleep(0.1)
else:
    raise RuntimeError("Uvicorn did not start on http://127.0.0.1:8000")

INFO:     Started server process [92335]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


In [9]:
import requests

response = requests.get("http://127.0.0.1:8000/")

print(response.status_code)
print(response.json())

INFO:     127.0.0.1:59497 - "GET / HTTP/1.1" 200 OK
200
{'message': 'Welcome to the FastAPI!'}


In [10]:
# response = requests.get(
#     "http://127.0.0.1:8000/get_data"
# )

# print(response.status_code)
# response.raise_for_status()
# print(response.json())

In [11]:
response = requests.get(
    "http://127.0.0.1:8000/get_data/1855"
)

print(response.status_code)
# response.raise_for_status()
print(response.json())

INFO:     127.0.0.1:59503 - "GET /get_data/1855 HTTP/1.1" 200 OK
200
{'id': 1855, 'gender': 'Men', 'base_color': 'Grey', 'product_display_name': 'Inkfruit Mens Chain Reaction T-shirt'}


In [12]:
response = requests.post(
    "http://127.0.0.1:8000/add_data",
    json={
        "id": 9999,
        "gender": "Men",
        "master_category": "Apparel",
        "article_type": "Shirt",
        "base_color": "Blue",
        "season": "Summer",
        "year": 2024,
        "usage": "Casual",
        "product_display_name": "Blue Summer Shirt"
    }
)

print(response.status_code)
response.raise_for_status()
print(response.json())

INFO:     127.0.0.1:59509 - "POST /add_data HTTP/1.1" 200 OK
200
{'message': 'Data added successfully'}
